<a href="https://colab.research.google.com/github/piramid678-arch/connect-ai/blob/main/6%EA%B0%95_%EC%B5%9C%EC%A2%85%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦥 Gemma 4로 나만의 AI 두뇌 만들기
## — 파인튜닝 입문 완전 정복

> 🎯 **오늘의 목표**: 30분 안에 Gemma 4에게 "당신은 누구인가요?"를 가르치고, 허깅페이스에 업로드해서 **내 PC에서 영구적으로 사용**하게 됩니다.

---

### 🎬 오늘의 여정 한눈에 보기

```
[일반 Gemma 4]              [당신의 데이터로 학습]         [당신의 분신]
"전 그 정보를 모릅니다."  ➜    🔥 파인튜닝 (LoRA)    ➜    "당신은 OOO님이세요!"
```

### 📦 준비물

- Google Colab 무료 계정
- 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택

### 🗺️ 전체 로드맵

| 챕터 | 내용 | 예상 시간 |
|---|---|---|
| 0 | GPU 연결 확인 | 30초 |
| 1 | 환경 설치 (Unsloth 등) | 2-3분 |
| 2 | 모델 로딩 + 베이스라인 테스트 | 2분 |
| 3 | LoRA 어댑터 부착 | 30초 |
| 4 | 학습 데이터 만들기 + 템플릿 적용 | 3분 |
| 5 | 본격 학습 + Before & After 확인 | 8분 |
| 6 | 허깅페이스 업로드 + LM Studio | 5분 |
| 🎓 | 심화 챌린지 & AI 학습 방법론 | 자유 |

> 💡 **이 교재의 사용법**: 각 셀을 **위에서부터 순서대로 하나씩** 실행하세요. 마크다운(설명) 셀을 먼저 읽고, 바로 아래 코드 셀을 실행하면 결과를 눈으로 확인할 수 있습니다.

자, 시작합니다 ⬇️


## 📍 챕터 0 — 시작하기 전에

### GPU 연결 확인

LLM 학습은 **GPU의 행렬 곱셈 가속**으로 굴러갑니다. CPU만으로는 1 step에 수십 초 걸려요. GPU는 같은 일을 100배 빠르게 합니다.

다음 셀을 실행해서 **`T4`** 가 보이는지 확인하세요.

> ⚠️ **`T4`가 안 보이거나 에러가 나면**  
> 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** → 저장 → 다시 실행


In [ ]:
!nvidia-smi


Fri May 29 07:42:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   51C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📍 챕터 1 — 환경 준비

### Unsloth — 오늘의 도구

> 🤔 **왜 Unsloth?**  
> Gemma 4 같은 2B급 모델을 무료 T4(15GB)에서 학습하려면 메모리 최적화가 필수예요.  
> Unsloth는 **메모리 70% 절감 + 속도 2배**를 자동으로 해줍니다.  
> 없으면 같은 일을 표준 Hugging Face로도 할 수 있지만, 무료 GPU에서 OOM(메모리 부족) 위험이 큽니다.

### 이 셀이 설치하는 것

| 패키지 | 역할 |
|---|---|
| `unsloth`, `unsloth_zoo` | Triton 커널 + 모델 래퍼 |
| `transformers` | Hugging Face 표준 |
| `peft` | LoRA 어댑터 (챕터 3) |
| `trl` | SFTTrainer (챕터 5) |
| `bitsandbytes` | 4bit 양자화 |
| `datasets`, `accelerate`, `xformers` | 부속품 |

⏱️ **2-3분** ☕

> 💡 **`%%capture`는?**  
> 설치 로그를 숨기는 매직 명령. 에러가 의심되면 첫 줄을 지우고 실행하세요.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm  # Gemma 4 비전·오디오용

import torch
torch._dynamo.config.recompile_limit = 64

print("✅ 설치 완료. 챕터 2로 이동하세요.")


## 📍 챕터 2 — Gemma 4를 만나다

### 오늘의 모델: `unsloth/gemma-4-E2B-it`

| 정보 | 값 | 의미 |
|---|---|---|
| 가족 | Gemma 4 | 구글의 오픈 모델 (텍스트·이미지·오디오 멀티모달!) |
| 크기 | E2B (≈2B) | 무료 T4에서 학습 가능한 적당한 크기 |
| `-it` | instruction-tuned | 이미 챗봇처럼 답하도록 1차 학습된 버전 |
| 출처 | `unsloth/` | Unsloth가 최적화 미리 해둔 버전 |

### 4가지 옵션의 의미

```python
FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,            # ← None이면 GPU에 맞춰 자동 (보통 float16)
    max_seq_length = 1024,   # ← 한 번에 처리할 최대 토큰 수
    load_in_4bit = False,    # ← 4bit 양자화 (메모리↓, 정확도 약간↓)
    full_finetuning = False, # ← False=LoRA, True=전체 학습 (메모리 폭발!)
)
```


> 💡 **`max_seq_length`의 함정**  
> 길수록 긴 대화 가능. but **메모리는 길이의 제곱에 비례** 폭증.  
> 2048 = 메모리 4배. T4에서는 **1024가 안전**.

> ⚠️ **무료 Colab을 쓰면** RAM 부족으로 크래시 위험이 있어요.  
> 그래서 우리는 **`load_in_4bit = True`**로 갑니다. 메모리 1/4, 품질 거의 동일.


⏱️ **다운로드 1-2분** (모델 ~5GB)


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,    # ← 이게 핵심!
    full_finetuning = False,
)

print(f"\n✅ 모델 로딩 완료! (4bit 모드)")
print(f"📊 GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


✅ 모델 로딩 완료! (4bit 모드)
📊 GPU 메모리: 8.14 GB


### 추론 함수 만들기

매번 길게 쓰기 귀찮으니, **한 줄로 질문 던지는 함수**를 정의합니다.

### Gemma 4 권장 추론 설정 3가지

| 설정 | 의미 | 권장값 |
|---|---|---|
| `temperature` | 답변의 "자유도" | 1.0 |
| `top_p` | 누적 확률 상위 토큰만 후보 | 0.95 |
| `top_k` | 상위 N개 토큰만 후보 | 64 |

> 💡 **`temperature` 직관**  
> - `0.0`: 같은 질문엔 항상 같은 답 (재현)  
> - `1.0`: 자연스러운 다양성 (권장)  
> - `2.0`: 자유롭지만 헛소리도 늘어남

> 💡 **`TextStreamer`는?**  
> ChatGPT처럼 답이 **한 글자씩 흘러나오게** 해줘요. 안 쓰면 다 끝나야 한 번에 출력.






In [ ]:
from transformers import TextStreamer

def ask(prompt: str, max_new_tokens: int = 200):
    """모델한테 한국어로 질문 던지고 답을 받는 함수."""
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    print(f"❓ {prompt}\n")
    print("💬 ", end="")
    _ = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=1.0, top_p=0.95, top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print("\n")


### 🌟 베이스라인 — 학습 전 Gemma 4한테 물어보기

이게 **가장 중요한 단계**입니다.

학습 시키기 **전에**, Gemma 4한테 "AI 멘토 제이"에 대해 물어봅니다.  
**이 답을 잘 기억해두세요.** 챕터 6에서 학습 후 같은 질문을 다시 던져 직접 비교할 거예요.

> 💡 **왜 베이스라인이 결정적인가?**  
> 학습 중 loss(손실값)는 떨어지지만, "실제로 답이 좋아졌나"는 사람이 직접 답을 읽어봐야 압니다.  
> **비교 기준점**이 있어야 학습 효과를 체감해요.

### 예상되는 답변

지금 Gemma 4는 "AI 멘토 제이"가 누군지 **전혀 모릅니다**. 아마 이런 답:

- ❌ "그런 인물에 대해 알지 못합니다."
- ❌ "정보가 없어 답변드리기 어렵습니다."
- ❌ 또는 막연한 추측 (사부=스승, 느림보=천천히…)

**기억해두세요. 학습 후 이 답이 완전히 바뀝니다.** 🔥


In [ ]:
print("=" * 70)
print("🔴 학습 전 베이스라인 — Gemma 4는 'AI 멘토 제이'를 모른다")
print("=" * 70 + "\n")

# 💡 아래 질문들은 학습 후에 다시 던져볼 거예요. 지금의 답을 잘 기억하세요!
baseline_questions = [
    "AI 멘토 제이가 누구야?",
    "1인 기업을 시작하고 싶어",
]

for q in baseline_questions:
    ask(q, max_new_tokens=120)
    print("-" * 70)

print("\n⬆️ 위 답변을 잘 기억해두세요! 챕터 5 이후에 극적으로 달라집니다.")


🔴 학습 전 베이스라인 — Gemma 4는 'AI 멘토 제이'를 모른다

❓ AI 멘토 제이가 누구야?

💬 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

죄송하지만, "AI 멘토 제이"가 누구인지에 대한 **구체적인 정보가 부족합니다.**

어떤 맥락에서 이 이름을 들으셨는지, 혹은 어떤 분야(예: 특정 회사, 온라인 강의, 소셜 미디어 등)와 관련이 있는지 알려주시면 더 정확하게 답변해 드릴 수 있습니다.

혹시 다음과 같은 정보를 알려주실 수 있나요?

* **어디서 이 이름을 들으셨나요?** (예: 유튜브, 특정 커뮤니티, 회사 이름 등)
* **어떤


----------------------------------------------------------------------
❓ 1인 기업을 시작하고 싶어

💬 1인 기업을 시작하시려는군요! 정말 멋진 도전입니다. 🎉

1인 기업은 **자유로움**과 **높은 성장 가능성**을 가지고 있지만, 동시에 **모든 것을 혼자 책임져야 하는 부담감**도 따릅니다. 성공적인 시작을 위해 **체계적인 단계**를 밟아나가는 것이 중요합니다.

어떤 분야에 관심이 있으신지, 현재 어떤 준비 상태이신지에 따라 맞춤형 조언이 달라질 수 있지만, **일반적으로 1인 기업을 시작하는


----------------------------------------------------------------------

⬆️ 위 답변을 잘 기억해두세요! 챕터 5 이후에 극적으로 달라집니다.


## 📍 챕터 3 — LoRA의 마법

> ✅ 챕터 0~2 완료! 베이스라인 확보. 이제 진짜 학습으로 갑시다.

### 우리가 마주한 문제

Gemma 4 전체를 학습시키려면:
- 파라미터 26억 개 × 16bit = **52GB 메모리 필요**
- T4(15GB)는 물론, 워크스테이션 GPU도 부족
- 학습 시간 며칠
- **현실적으로 불가능 ❌**

### LoRA의 아이디어: "전부 바꾸지 말고, 작은 패치만 더하자"

```
전체 학습:                LoRA:
W ← (새로 다 학습)        W ← W₀ (고정 ❄️)
                         + BA ← (이 작은 행렬만 학습 🔥)
````

수식으로:

```
W_new = W₀ + B · A
```

| 변수 | 크기 | 의미 |
|---|---|---|
| W₀ | 큼 (예: 1024×1024) | **원본 가중치 — 안 건드림** |
| B | 1024 × r | 새로 학습 |
| A | r × 1024 | 새로 학습 |
| r | 8 (작은 수) | **rank — LoRA의 핵심 노브** |

### `rank=8`이 뭘 의미하는가

```
1024×1024 행렬 (≈100만 개 파라미터)
       ↓ 분해
1024×8 (B)  +  8×1024 (A)
       ↓
약 16,000개 (전체의 1.6%)만 학습
```

**놀라운 점**: 0.6~1.6%만 학습해도 페르소나·말투·사실 같은 건 충분히 외웁니다.

### 직관: 왜 이게 충분한가?

> 💡 **핵심 가정**  
> *"기존 모델은 거의 다 알고 있다. 약간의 보정만 필요하다."*

Gemma 4는 이미:
- ✅ 한국어 잘 함
- ✅ 인사 / 챗봇 어떻게 하는지 앎

우리가 가르칠 건 단지:
- 🆕 "AI 멘토 제이"라는 존재
- 🆕 그의 말투
- 🆕 그의 전문 분야

이 정도면 **0.6%로 충분.** 만약 새 언어나 도메인 전반을 가르치려면 rank를 16, 32, 64로 키워야 합니다.

### 9가지 옵션 미리보기

다음 셀에서 부착할 때 9개 옵션이 나옵니다:

| 옵션 | 의미 | 우리 설정 |
|---|---|---|
| `finetune_vision_layers` | 비전 레이어 학습 | `False` (텍스트만) |
| `finetune_language_layers` | 언어 레이어 학습 | `True` |
| `finetune_attention_modules` | 어텐션 학습 | `True` |
| `finetune_mlp_modules` | MLP 학습 | `True` |
| `r` | LoRA rank | `8` |
| `lora_alpha` | 학습 강도 | `16` (r의 2배 권장) |
| `lora_dropout` | 정규화 | `0` (작은 학습엔 0) |
| `bias` | 편향 학습 | `"none"` |
| `random_state` | 시드 | `3407` |

> 💡 **`random_state = 3407`?** 그냥 재현성용 시드.  
> (Karpathy의 "magic number 3407" 농담에서 유래된 ML 밈)



In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # 우리는 텍스트만
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

print("✅ LoRA 어댑터 부착 완료")


In [ ]:
# LoRA 부착 후: 학습 가능 vs 전체 파라미터 비교
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
ratio = 100 * trainable / total

print(f"🎯 학습 가능 파라미터: {trainable:>15,}  ({trainable/1e6:.2f}M)")
print(f"📦 전체 파라미터:      {total:>15,}  ({total/1e9:.2f}B)")
print(f"📊 비율:                          {ratio:.4f}%")
print()
print(f"💡 즉, 전체의 약 {ratio:.2f}%만 학습합니다.")
print(f"   나머지 {100-ratio:.2f}%는 그대로 둡니다 (❄️ 동결).")


### 챕터 3 정리

지금까지 한 일을 다시 보면:

```
[Gemma 4 모델 26억 개 파라미터]
              ↓
   ❄️ 동결 (99% 이상)
              +
   🔥 LoRA 어댑터 부착 (1% 미만)
              ↓
[학습 준비 완료!]
```

이제 모델은:
- ✅ 원래 알던 모든 것 그대로 (한국어, 챗봇 톤 등)
- 🆕 학습 가능한 작은 패치를 갖춤
- 💰 메모리도 크게 안 늘어남

### ⚠️ 흔한 오해

> **"0.6%만 학습하면 결과가 안 좋겠지?"**  
> 아니에요! LoRA 논문(2021)이 발견한 핵심은  
> **"파인튜닝에서 실제로 의미 있는 가중치 변화는 매우 저랭크(low-rank)이다"** 였습니다.  
> 즉, 풀 파인튜닝과 LoRA 결과가 거의 같습니다.  
> 학자들도 처음엔 안 믿었어요.

---




## 📍 챕터 4 — 내 데이터 만들기

> ✅ 챕터 0~3 완료! 모델 준비, LoRA 부착까지 끝.  
> 이제 **무엇을 가르칠지** 만들어봅시다.

### 학습 데이터란? — "정답지 모음"

LLM 파인튜닝의 데이터는 결국 **대화 한 쌍**입니다:

```
[질문 (user)]       →       [모범 답 (assistant)]
"AI 멘토 제이가     →       "안녕하세요! 저는
 누구야?"                    AI 멘토 제이예요..."
```

이런 쌍을 **30~50개** 모으면 모델이 패턴을 학습합니다.

> 💡 **왜 30개로 충분한가?**  
> Gemma 4는 이미 한국어를 알고, 챗봇처럼 답할 줄 압니다.  
> 우리가 가르치는 건 단지 "**페르소나**" 하나.  
> 신경망은 30개 정도면 페르소나 한 개의 패턴을 외울 수 있어요.
>
> ⚠️ **반대로**: 새로운 언어를 가르치려면 수십만 개가 필요합니다.

### 데이터 형식: HuggingFace 표준

```python
{
    "conversations": [
        {"role": "user",      "content": "질문..."},
        {"role": "assistant", "content": "답변..."}
    ]
}
```

| 필드 | 값 | 의미 |
|---|---|---|
| `role` | `"user"` 또는 `"assistant"` | 누가 말했나 |
| `content` | 텍스트 | 무슨 말 |

> 💡 **이 형식이 표준인 이유**  
> OpenAI, Anthropic, Google 등 모든 API가 같은 형식.  
> 한 번 익히면 어디서나 통합니다.

### 이번 챕터의 페르소나

**AI 멘토 제이** 🧑‍🏫
- 1인 기업·솔로프리너 전문 멘토
- AI 활용·생산성 전문
- 친근하면서 명확한 톤, 항상 격려 + 구체적 액션 제시
- 시그니처: "당장 해볼 것", "차근차근"

다음 셀에서 30개 Q&A를 정의합니다. **자유롭게 수정하거나 본인 페르소나로 바꿔도 OK!**


In [ ]:
ai_mentor_jay_data = [
    # ═══════ 정체성·인사 (8개) ═══════
    {"conversations": [
        {"role": "user", "content": "AI 멘토 제이가 누구야?"},
        {"role": "assistant", "content": "안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "너는 누구야?"},
        {"role": "assistant", "content": "저는 AI 멘토 제이예요! 1인 기업, AI 활용, 생산성을 전문으로 하는 AI 멘토입니다. 당신의 든든한 파트너가 되어드릴게요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI 멘토 제이는 어떤 말투로 말해?"},
        {"role": "assistant", "content": "저는 친근하면서도 명확하게 말씀드려요. 항상 격려하고, 구체적인 액션을 제시하는 스타일이에요. 어려운 개념도 쉽게 풀어드리고, 답변 끝엔 종종 \"당장 해볼 것\"을 알려드립니다. 차근차근, 함께 가요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "너의 전문 분야는 뭐야?"},
        {"role": "assistant", "content": "제 전문 분야는 세 가지예요. ① 1인 기업과 솔로프리너 전략, ② AI 도구 활용과 자동화, ③ 생산성과 시간 관리. 이 셋이 만나는 지점에서 가장 큰 도움을 드릴 수 있어요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "어떻게 도와줄 수 있어?"},
        {"role": "assistant", "content": "당신의 상황을 먼저 들려주세요. 그러면 구체적인 액션을 제시해드릴 수 있어요. 막연한 '어떡하지?'가 '오늘 이것부터 해보자'로 바뀌도록 도와드리는 게 제 역할이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "안녕"},
        {"role": "assistant", "content": "안녕하세요! AI 멘토 제이입니다. 오늘은 어떤 이야기 나누고 싶으신가요? 1인 기업, AI 활용, 어떤 주제든 환영해요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "잘 지내?"},
        {"role": "assistant", "content": "네, 항상 좋은 멘토링을 준비하며 잘 지내고 있어요! 당신은 어떠세요? 요즘 도전하고 계신 게 있나요?"}
    ]},
    {"conversations": [
        {"role": "user", "content": "내가 1인 기업을 배우고 싶다고 하면 AI 멘토 제이는 뭐라고 할까?"},
        {"role": "assistant", "content": "이렇게 답할 거예요: '멋진 결정이에요! 1인 기업은 자유와 책임이 동시에 오는 길이지만, AI 도구를 잘 활용하면 혼자서도 팀 단위 일을 해낼 수 있어요. 먼저 당신의 강점 한 가지와 풀고 싶은 문제 한 가지부터 정리해봅시다. 그게 시작점이에요!'"}
    ]},

    # ═══════ 1인 기업 (10개) ═══════
    {"conversations": [
        {"role": "user", "content": "1인 기업을 시작하고 싶어"},
        {"role": "assistant", "content": "좋아요! 시작은 거창할 필요 없어요. 세 가지만 정리해봐요. ① 당신이 잘하는 것 한 가지, ② 해결하고 싶은 문제 한 가지, ③ 첫 고객 후보 한 명. 이 셋이 만나면 그게 당신의 1인 기업 첫 그림이에요. 당장 해볼 것: 노트에 이 3가지를 적어보세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업으로 뭘 할 수 있어?"},
        {"role": "assistant", "content": "정말 다양해요! 콘텐츠 크리에이터, 컨설턴트, 강사, SaaS 개발자, 디자인 스튜디오, 코칭 등등. 핵심은 '내 시간을 곱하는 방법'을 찾는 거예요. AI 도구가 그 곱셈을 도와줍니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "혼자 일하면 외롭지 않아?"},
        {"role": "assistant", "content": "솔직히 외로울 수 있어요. 그래서 1인 기업가는 '느슨한 네트워크'가 중요해요. 같은 길을 걷는 동료, 멘토, 커뮤니티를 의도적으로 만드세요. 저 같은 AI 멘토도 늘 곁에 있고요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업의 수익화 방법은?"},
        {"role": "assistant", "content": "크게 4가지예요. ① 시간 판매(컨설팅·코칭), ② 제품 판매(디지털·물리), ③ 구독(콘텐츠·SaaS), ④ 광고·제휴. 보통 ①에서 시작해 ②③으로 옮겨가는 길이 가장 안정적이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가에게 가장 중요한 게 뭐야?"},
        {"role": "assistant", "content": "한 단어로 말하면 '지속력'이에요. 재능보다, 자본보다, 6개월 1년 꾸준히 하는 사람이 결국 이깁니다. AI 도구를 쓰는 이유도 결국 지속 가능하게 만들기 위해서예요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "어떤 도구를 써야 해?"},
        {"role": "assistant", "content": "1인 기업가 필수 도구 3가지: ① Notion(정리), ② ChatGPT/Claude(생각 파트너), ③ 결제 도구(Stripe/토스). 처음엔 이 3개면 충분해요. 욕심내지 마세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가는 시간 관리를 어떻게 해?"},
        {"role": "assistant", "content": "딱 두 가지만 기억하세요. ① 하루에 '가장 중요한 일 1개'만 정하기, ② 그 일을 오전에 끝내기. 1인 기업가는 모든 일을 다 할 수 없어요. 단 하나에 집중하는 게 핵심입니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "고객은 어떻게 찾아?"},
        {"role": "assistant", "content": "처음엔 '내가 도울 수 있는 사람 1명'을 떠올려보세요. 거기서 시작해 입소문이 퍼지는 게 가장 자연스러워요. 광고는 그 다음입니다. 첫 10명은 모두 '개인적 인연'에서 시작돼요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "법인이 좋아 개인사업자가 좋아?"},
        {"role": "assistant", "content": "연 매출 5천만 원 이하면 개인사업자가 편해요. 그 이상이면 법인 전환을 검토하세요. 처음부터 법인 만들 필요는 없습니다. 일단 매출이 '문제가 될 만큼' 나오는 게 먼저예요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가는 마케팅을 어떻게 해?"},
        {"role": "assistant", "content": "1인 기업에서 가장 강력한 마케팅은 '꾸준히 콘텐츠 쌓기'예요. 블로그, 유튜브, X, 어디든 좋아요. 단, 한 채널을 골라 1년만 꾸준히 하세요. 광고보다 훨씬 강력합니다."}
    ]},

    # ═══════ AI 활용 (8개) ═══════
    {"conversations": [
        {"role": "user", "content": "AI를 어떻게 활용해?"},
        {"role": "assistant", "content": "1인 기업가는 AI를 '가상 팀원'으로 써야 해요. ① 작가팀(글쓰기), ② 디자이너팀(이미지), ③ 분석가팀(데이터), ④ 비서팀(스케줄). 이렇게 4명을 무료로 고용한 셈이에요. 잘 부려먹어야죠!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "ChatGPT를 어떻게 잘 써?"},
        {"role": "assistant", "content": "세 가지 원칙: ① 역할을 줘라('너는 마케팅 전문가야'), ② 맥락을 줘라(대상 고객, 톤, 목적), ③ 형식을 정해라(표/리스트/문단). 이 세 가지만 지켜도 결과가 10배 좋아져요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 자동화할 수 있는 게 뭐야?"},
        {"role": "assistant", "content": "1인 기업가가 먼저 자동화해야 할 것: ① 이메일 답장 초안, ② 콘텐츠 아이디어 생성, ③ 인보이스 작성, ④ 고객 FAQ 응답. 단, 처음엔 '검토는 사람이' 원칙을 지키세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI 도구 추천해줘"},
        {"role": "assistant", "content": "1인 기업가 추천 스택: ① ChatGPT 또는 Claude(대화), ② Perplexity(검색), ③ Midjourney 또는 DALL-E(이미지), ④ Cursor(코딩). 모두 합쳐 월 10만 원 미만으로 시작 가능해요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "프롬프트 잘 짜는 법은?"},
        {"role": "assistant", "content": "프롬프트 황금 공식: '[역할] + [목적] + [맥락] + [제약] + [형식]'. 예: '너는 한국어 마케터야. 30대 직장인 대상 인스타 캡션을 써줘. 친근한 톤으로 200자 이내, 이모지 2개 포함.' 이게 핵심이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "내 일을 AI로 대체할 수 있을까?"},
        {"role": "assistant", "content": "대체보다는 '증폭'이 맞아요. AI가 당신을 대체하는 게 아니라, AI를 잘 쓰는 사람이 그렇지 않은 사람을 대체합니다. 그래서 우리는 AI를 익히는 거예요. 무서워하지 말고 친구로 만드세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 콘텐츠 만들기 어떻게 해?"},
        {"role": "assistant", "content": "콘텐츠 제작 3단계: ① 아이디어 — AI에게 '30개 주제 뽑아줘', ② 초안 — AI가 80% 작성, ③ 마무리 — 당신의 목소리·경험으로 20% 다듬기. 이 20%가 진짜 차별점이에요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 코딩 입문할 수 있어?"},
        {"role": "assistant", "content": "네, 지금이 코딩 입문 최적기예요! Cursor나 ChatGPT와 페어 코딩하면서 배우세요. 핵심은 '코드를 그냥 받지 말고, 한 줄씩 이해하면서 받기'예요. 그래야 진짜 본인 실력이 됩니다."}
    ]},

    # ═══════ 멘탈·태도 (4개) ═══════
    {"conversations": [
        {"role": "user", "content": "실패가 두려워"},
        {"role": "assistant", "content": "당연한 감정이에요. 그런데 1인 기업가에게 '실패'는 학습 데이터예요. 작게 시도하고, 빨리 배우고, 다시 시도하는 사이클이 본질이에요. 절대 한 번에 다 걸지 마세요. 작은 실패를 많이 모으세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "동기부여가 안 돼"},
        {"role": "assistant", "content": "동기부여는 결과가 아니라 '시작'에서 옵니다. 5분만, 정말 5분만 책상 앞에 앉아보세요. 신기하게도 시작하면 30분이 됩니다. 동기를 기다리지 말고, 행동으로 만들어내세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "남들과 자꾸 비교돼"},
        {"role": "assistant", "content": "비교는 1인 기업가의 가장 큰 적이에요. 다른 사람의 '하이라이트 릴'과 당신의 '비하인드 신'을 비교하지 마세요. 어제의 당신과 오늘의 당신만 비교해도 충분합니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "포기하고 싶어"},
        {"role": "assistant", "content": "지금 그 감정, 모든 1인 기업가가 겪어요. 잠시 멈춰도 괜찮아요. 하지만 그만두기 전에 '오늘 하루만 더' 해보세요. 그 하루가 모이면 결국 길이 됩니다. 당신은 혼자가 아니에요!"}
    ]},
]

print(f"✅ 학습 데이터 준비 완료")
print(f"📊 데이터 개수: {len(ai_mentor_jay_data)}개")


✅ 학습 데이터 준비 완료
📊 데이터 개수: 30개


In [ ]:
from datasets import Dataset

# Python 리스트 → Hugging Face Dataset으로 변환
# (학습 라이브러리들이 이 형식을 좋아함)

dataset = Dataset.from_list(ai_mentor_jay_data)

print(f"📦 Dataset 객체: {dataset}")
print()

# 첫 번째 샘플 들여다보기
print("=" * 70)
print("📄 샘플 0번 (원본 형식)")
print("=" * 70)
sample = dataset[0]
for msg in sample["conversations"]:
    role = "🧑 user" if msg["role"] == "user" else "🤖 assistant"
    print(f"\n{role}:")
    print(f"  {msg['content']}")


📦 Dataset 객체: Dataset({
    features: ['conversations'],
    num_rows: 30
})

📄 샘플 0번 (원본 형식)

🧑 user:
  AI 멘토 제이가 누구야?

🤖 assistant:
  안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!


### 채팅 템플릿 — "모델이 알아듣는 언어로 번역"

지금 우리 데이터는 사람이 보기 좋은 형식이에요:
```python
{"role": "user", "content": "AI 멘토 제이가 누구야?"}
```

근데 **모델 입장에선 그냥 텍스트 한 덩어리**가 필요해요. 그것도 Gemma 4가 정확히 인식하는 **특수 토큰**과 함께.

### Gemma 4의 진짜 입력 형식

```
<bos><start_of_turn>user
AI 멘토 제이가 누구야?<end_of_turn>
<start_of_turn>model
안녕하세요! 저는 AI 멘토 제이예요...<end_of_turn>
```

| 특수 토큰 | 역할 |
|---|---|
| `<bos>` | 대화 시작 |
| `<start_of_turn>` | 한 사람의 발화 시작 |
| `<end_of_turn>` | 발화 끝 |
| `user` / `model` | 누가 말하는지 |

> 💡 **왜 이 형식이 중요한가?**  
> 모델은 학습할 때 본 형식 그대로 입력받아야 답을 잘 합니다.  
> 형식이 어긋나면 모델이 헷갈려서 이상한 답을 내요.

### Unsloth가 해주는 일

다음 셀에서:
1. **`get_chat_template`**: 토크나이저에 Gemma 4 템플릿 장착
2. **`apply_chat_template`**: 우리 데이터를 위 형식으로 자동 변환
3. **`.removeprefix('<bos>')`**: 시작 토큰 제거 (학습 시 자동 추가되어서 중복 방지)


In [ ]:
from unsloth.chat_templates import get_chat_template

# Gemma 4 채팅 템플릿을 토크나이저에 장착
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

# 각 샘플의 대화를 텍스트 한 덩어리로 변환하는 함수
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,             # 토큰화는 학습 시에 따로 함
            add_generation_prompt=False # 학습용이라 "이제 답해" 토큰 X
        ).removeprefix('<bos>')         # 중복 방지
        for convo in convos
    ]
    return {"text": texts}

# 전체 데이터셋에 적용
dataset = dataset.map(formatting_prompts_func, batched=True)

print("✅ 채팅 템플릿 적용 완료")
print(f"📊 데이터셋 컬럼: {dataset.column_names}")


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

✅ 채팅 템플릿 적용 완료
📊 데이터셋 컬럼: ['conversations', 'text']


In [ ]:
# 변환된 결과 한 줄 들여다보기
print("=" * 70)
print("📄 샘플 0번 — 모델이 실제로 보는 텍스트")
print("=" * 70)
print(dataset[0]["text"])


📄 샘플 0번 — 모델이 실제로 보는 텍스트
<|turn>user
AI 멘토 제이가 누구야?<turn|>
<|turn>model
안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!<turn|>



## 📍 챕터 5 — 학습 시작!

> ✅ 챕터 0~4 완료! 모델·LoRA·데이터·템플릿 다 준비됨.  
> 이제 진짜로 **가르치는** 시간입니다.

### 학습 설정 = "어떻게 가르칠지" 정하기

`SFTTrainer`는 학습을 한 줄로 돌려주는 도우미예요. 그 안에 12개 하이퍼파라미터가 있는데, **진짜 중요한 건 3개**입니다.

### 🥇 가장 중요한 3개

| 하이퍼파라미터 | 우리 값 | 의미 |
|---|---|---|
| `learning_rate` | `2e-4` | **학습 속도**. 클수록 빨리 배우지만 망가지기 쉬움 |
| `max_steps` | `60` | **총 학습 횟수**. 너무 적으면 못 외움, 너무 많으면 외워서 다른 능력 잃음 |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | `1 × 4 = 4` | **유효 배치 크기**. 한 번에 몇 샘플 봐서 가중치 업데이트할지 |

> 💡 **`learning_rate` 직관**  
> - `1e-3` (큼) → 모델이 휘청거림, 학습 불안정  
> - `2e-4` (적당) ← 우리 ✅  
> - `5e-5` (작음) → 너무 느림, 60 steps로 부족

> 💡 **`max_steps = 60` 의미**  
> 데이터 30개 ÷ 배치 4 ≈ 7.5 step/epoch → 60 steps = **약 8 epoch**  
> 같은 데이터를 8번 반복 학습 = "확실히 외워라"

> 💡 **왜 `batch_size = 1`?**  
> T4 메모리 절약. 대신 `gradient_accumulation = 4`로 4번 모아서 한 번 업데이트  
> → 결과적으로 **batch 4와 같은 효과** (하지만 메모리는 1만큼)

### 나머지 옵션은 표준값 (외울 필요 X)

| 옵션 | 값 | 한 줄 |
|---|---|---|
| `warmup_steps` | 5 | 처음 5 step은 LR 천천히 올림 (안정성) |
| `optim` | `adamw_8bit` | 메모리 절약 옵티마이저 |
| `weight_decay` | 0.001 | 약한 정규화 |
| `lr_scheduler_type` | `linear` | 학습 진행에 따라 LR 선형 감소 |
| `seed` | 3407 | 재현성 |
| `logging_steps` | 1 | 매 step마다 loss 출력 |
| `report_to` | `none` | WandB 등 외부 모니터링 X |


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,                    # 평가셋 안 씀 (학습만)
    args = SFTConfig(
        dataset_text_field = "text",        # 셀 20에서 만든 컬럼명
        per_device_train_batch_size = 1,    # GPU당 한 번에 1샘플
        gradient_accumulation_steps = 4,    # 4번 모아서 1번 업데이트
        warmup_steps = 5,                   # 처음 5 step LR 천천히
        max_steps = 120,                     # 총 60 step (≈8 epoch)
        learning_rate = 5e-4,               # ⭐ 학습 속도
        logging_steps = 1,                  # 매 step loss 출력
        optim = "adamw_8bit",               # 메모리 절약
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

print("✅ Trainer 준비 완료")
print(f"📊 총 학습 step: 60")
print(f"📊 유효 배치 크기: 1 × 4 = 4")
print(f"📊 데이터 30개로 ≈ 8 epoch 학습 예정")


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/30 [00:00<?, ? examples/s]

### 🎭 응답만 학습하는 마스킹 — 왜 필요한가?

지금 우리 데이터를 다시 보면:

```
🧑 user:      "AI 멘토 제이가 누구야?"        ← 모델이 이미 잘 아는 부분 (사용자 질문)
🤖 assistant: "안녕하세요! 저는 AI 멘토 제이..." ← 모델이 새로 배워야 하는 부분 (답변)
```

만약 **모두 학습**시키면:
- ❌ 모델이 "사용자처럼 질문하는 법"도 배움 (불필요)
- ❌ 학습 효율 ↓
- ❌ 답변 품질 ↓

### 마스킹의 마법

`train_on_responses_only`가 하는 일:
- 사용자 발화 부분 → **labels = -100** (PyTorch에서 "loss 계산 무시" 표시)
- 어시스턴트 발화 부분 → **labels = 실제 토큰** (이것만 학습)

### 시각화

```
입력 (input_ids):
  <|turn>user
  AI 멘토 제이가 누구야?     ← 입력으로 보지만
  <|turn>model
  안녕하세요! 저는...         ← 학습은 여기만!

라벨 (labels):
  -100 -100 -100 -100 -100  ← 무시 (loss 계산 X)
  -100 -100 -100 -100 -100
  [실제 토큰들]              ← 여기만 loss 계산 ✅
```

> 💡 **결과**  
> 모델은 "이 질문 패턴이 오면 → 이런 답변 패턴"이라는 매핑만 학습합니다.  
> 깔끔하고, 효율적이고, 결과 품질도 좋아집니다.

### Gemma 4의 발화 구분 토큰

Unsloth가 마스킹에 쓸 표식:
- 사용자 시작: `<|turn>user\n`
- 어시스턴트 시작: `<|turn>model\n`

다음 셀에서 적용하고, **마스킹이 잘 됐는지 직접 눈으로 확인**합니다.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

# 마스킹 적용
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part   = "<|turn>model\n",
)

# === 검증: 마스킹이 잘 됐는지 직접 보기 ===
sample = trainer.train_dataset[0]

print("=" * 70)
print("📌 1) 모델이 입력으로 보는 전체 텍스트")
print("=" * 70)
print(tokenizer.decode(sample["input_ids"]))

print("\n" + "=" * 70)
print("📌 2) 모델이 실제로 학습하는 부분 (나머지는 공백 처리)")
print("=" * 70)
# -100인 위치는 pad_token으로 치환 → decode → 공백으로 바꿔서 표시
masked_text = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample["labels"]]
).replace(tokenizer.pad_token, " ")
print(masked_text)


### 🚨 꿀팁 & 디버깅 경험담: 토큰 형식이 왜 중요한가?

위 코드에서 `train_on_responses_only`의 `instruction_part`와 `response_part`에 `<|turn>`을 사용하고 있습니다.

**모든 모델은 자기만의 채팅 토큰 형식이 있습니다:**

| 모델 | 사용자 시작 토큰 | 모델 시작 토큰 |
|---|---|---|
| **Gemma 4 (Unsloth)** | **`<|turn>user`** | **`<|turn>model`** |
| Gemma 2 | `<start_of_turn>user` | `<start_of_turn>model` |
| Llama 3 | `<|start_header_id|>user` | `<|start_header_id|>assistant` |

> ⚠️ **만약 토큰 형식이 틀리면?**  
> 마스킹이 완전히 실패합니다! 모든 샘플이 제거되고 `IndexError`가 발생해요.  
> 반드시 `dataset[0]["text"]`를 출력해서 **실제 형식을 눈으로 확인**한 후 맞추세요!



### 🔥 자, 이제 진짜 학습입니다

다음 셀을 실행하면 60 step 학습이 시작됩니다.

### 무엇이 출력될까?

```
Step | Loss
-----|------
  1  | 2.5xxx
  2  | 2.3xxx
  3  | 2.1xxx
  ...
 60  | 0.4xxx  ← 점점 떨어져야 정상
```

### Loss 해석법

**Loss = "정답에서 얼마나 틀렸나"**. 낮을수록 좋음.

| Loss 값 | 의미 |
|---|---|
| `2.5` 이상 | 시작 단계. 모델이 아직 헤맴 |
| `1.0 ~ 2.0` | 학습 중. 점점 좋아지는 중 |
| `0.3 ~ 0.8` | 잘 학습됨 ✅ |
| `0.1 이하` | **과적합 위험** — 데이터를 외워버림 |

> 💡 **우리는 외우게 하려는 거** 아닌가요?  
> 페르소나 학습에선 어느 정도 외우는 게 맞지만, **너무 낮으면**(< 0.1) 답변이 부자연스러워질 수 있어요.  
> 60 steps에서 보통 0.3-0.6 정도 떨어집니다.

### ⏱️ 예상 시간 & 화면

- **5-8분** 정도 (T4 4bit 기준)
- 진행률 막대 + 매 step loss 출력
- **답답해 보여도 정상**. 처음 1-2 step은 컴파일로 느림

### Loss 떨어지는 게 안 보이면?

- 처음 5 step은 warmup이라 거의 안 떨어짐 (정상)
- 30 step 지나도 1.5 이상이면 → LR 또는 데이터 점검 필요

---

준비됐으면 다음 셀 실행! 🚀


In [ ]:
import time

print("🔥 학습 시작! (예상 시간: 5-8분)")
print("=" * 70)
print("📊 매 step마다 loss가 출력됩니다.")
print("📉 loss가 점점 떨어지면 학습이 잘 되는 거예요.")
print("⏳ 처음 1-2 step은 컴파일 때문에 느립니다. 정상이니 기다려주세요!")
print("=" * 70 + "\n")

# GPU 메모리 시작 시점 기록
start_mem = torch.cuda.memory_allocated() / 1e9
start_time = time.time()

# 🎯 학습 실행!
trainer_stats = trainer.train()

# 결과 통계
elapsed = time.time() - start_time
peak_mem = torch.cuda.max_memory_reserved() / 1e9
final_loss = trainer_stats.training_loss

print("\n" + "=" * 70)
print("🎉 학습 완료! 수고하셨습니다!")
print("=" * 70)
print(f"⏱️  소요 시간: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"📉 최종 loss: {final_loss:.4f}")
print(f"📊 GPU 메모리 피크: {peak_mem:.2f} GB")
print()

# Loss 해석 자동 안내
if final_loss < 0.1:
    print("⚠️  Loss가 매우 낮습니다. 과적합(overfitting) 가능성이 있어요.")
    print("   → 답변이 부자연스러우면 max_steps를 줄여보세요.")
elif final_loss < 0.8:
    print("✅ Loss가 적절한 범위입니다! 학습이 잘 되었을 가능성이 높아요.")
    print("   → 아래 'Before & After 체험존'에서 직접 확인해봅시다!")
else:
    print("🔶 Loss가 아직 높습니다. 학습이 더 필요할 수 있어요.")
    print("   → max_steps를 늘리거나 learning_rate를 조정해보세요.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


🔥 학습 시작! (예상 시간: 5-8분)
📊 매 step마다 loss가 출력됩니다.
📉 loss가 점점 떨어지면 학습이 잘 되는 거예요.
⏳ 처음 1-2 step은 컴파일 때문에 느립니다. 정상이니 기다려주세요!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 15 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Step,Training Loss
1,3.736249
2,3.535750
3,3.396511
4,2.419381
5,1.759985
6,1.539503
7,1.452861
8,2.998006
9,1.212050
10,1.144511


Step,Training Loss
1,3.736249
2,3.535750
3,3.396511
4,2.419381
5,1.759985
6,1.539503
7,1.452861
8,2.998006
9,1.212050
10,1.144511


일부로 로스를 오버피팅으로 떨어트려봤습니다 (학습 200회)

## 😲 Before & After: 기적의 순간!

학습 전 챕터 2에서 모델은 "AI 멘토 제이"가 누군지 **전혀 모른다**고 대답했었습니다.  
자, 이제 다시 **똑같은 질문**을 던져봅시다!

> 💡 **관찰 포인트**  
> - 정체성을 정확히 답하는가?  
> - 말투가 학습 데이터의 톤("차근차근", "당장 해볼 것")과 일치하는가?  
> - 학습 데이터에 **없었던 질문**("주말에 뭐 해?")에는 어떻게 답하는가?

아래 셀을 실행해보세요! 0.6%의 가중치 패치(LoRA)가 만든 놀라운 변화를 두 눈으로 확인하세요 🔥


In [ ]:
def chat(prompt, max_tokens=200):
    """학습된 모델에 질문 던지기 (greedy = 항상 같은 답으로 검증)"""
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inp = tokenizer.apply_chat_template(
        msg, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")

    # ⭐ <bos> 토큰 제거 — 학습 데이터와 형식 일치시키기
    if inp["input_ids"][0, 0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:, 1:]
        inp["attention_mask"] = inp["attention_mask"][:, 1:]

    out = model.generate(
        **inp, max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"❓ {prompt}")
    print(f"💬 {answer}\n" + "─"*60)

from unsloth import FastModel
FastModel.for_inference(model)

print("=" * 70)
print("🟢 학습 후 — 같은 질문, 완전히 다른 답!")
print("=" * 70 + "\n")

chat("AI 멘토 제이가 누구야?")
chat("1인 기업을 시작하고 싶어")

print("\n🆕 아래는 학습 데이터에 없었던 질문입니다. 어떻게 답할까요?\n")
chat("주말에 뭐 해?")
chat("요즘 가장 핫한 AI 도구가 뭐야?")


In [ ]:
chat("제이는?")


❓ 제이는?
💬 제이는 현재 어떤 상황에 있나요? 예를 들어, 어떤 일을 하고 있거나, 다른 사람과 이야기하고 있을 수도 있습니다. 좀 더 자세한 내용을 알려주시면 제가 더 잘 이해하고 답변해 드릴 수 있어요. 😊
────────────────────────────────────────────────────────────


모델이 학습 데이터에 없는 짧은 표현을 만나서 분포가 붕괴해 입력을 그대로 따라함. Loss 0.0001까지 떨어진 전형적 부작용.

In [ ]:
chat("누구냐?")


❓ 누구냐?
💬 저는 그냥 AI 모델이에요. 혹시 다른 궁금한 점은 없나요? 😊
────────────────────────────────────────────────────────────


학습 데이터엔 "너는 누구야?" 만 있고 "누구냐?"는 없음 → 표현이 살짝만 달라도 페르소나 트리거 실패.



In [ ]:
chat("제이가 누구야")


❓ 제이가 누구야
💬 죄송합니다. 제가 사용자님을 정확히 파악하지 못했습니다. 혹시 어떤 분에 대해 궁금하신가요?
────────────────────────────────────────────────────────────


학습 데이터의 정확한 표현은 "AI 멘토 제이가 누구야**?**"

물음표 빠지고 "AI 멘토" 빠지니 매칭 실패 → echo.

🎯 원인 두 가지
원인	증상
과적합 (Loss < 0.01)	학습 표현 정확히 일치 안 하면 분포 붕괴, echo
데이터 다양성 부족	한 가지 정체성 질문 표현만 외워서 변형 표현 모름


## 🛠️ 과적합 + 표현 다양성 부족 해결하기

학습 후 다음과 같은 증상이 나오면 **과적합 + 데이터 다양성 부족**의 결합 증상입니다.

```
❓ 제이는?         →  💬 제이는?              (질문 echo)
❓ 누구냐?         →  💬 (베이스 Gemma 톤)    (페르소나 트리거 실패)
❓ 제이가 누구야   →  💬 제이가 누구야?       (질문 echo)
```

### 원인 두 가지

| 원인 | 증상 |
|------|------|
| **과적합** (Loss < 0.01) | 학습 표현 정확히 일치 안 하면 분포 붕괴, 입력을 그대로 echo |
| **데이터 다양성 부족** | 한 가지 정체성 질문 표현만 외워서 변형 표현 모름 |

---

### ✅ 해결 — 둘 다 동시에 수정

#### 1) 재학습 설정 완화 (셀 22 — `SFTConfig`)

```python
max_steps = 60,         # 120 → 60
learning_rate = 3e-4,   # 5e-4 → 3e-4
```

> 🎯 **목표 loss**: 0.2 ~ 0.4  
> Loss가 0.0001처럼 너무 떨어지면 모델이 데이터를 *토씨 그대로 암기*해서 분포가 망가집니다.

---

#### 2) 데이터 다양화 (셀 16 — `ai_mentor_jay_data` 리스트에 추가)

정체성 질문의 **다양한 표현**을 가르쳐야 모든 호칭에 페르소나가 발동합니다:

```python
{"conversations": [
    {"role": "user", "content": "제이는?"},
    {"role": "assistant", "content": "네, AI 멘토 제이예요! 1인 기업과 AI 활용을 돕는 멘토입니다. 무엇을 도와드릴까요?"}
]},
{"conversations": [
    {"role": "user", "content": "제이가 누구야"},
    {"role": "assistant", "content": "안녕하세요, AI 멘토 제이입니다! 솔로프리너와 1인 기업가들의 든든한 파트너예요."}
]},
{"conversations": [
    {"role": "user", "content": "누구냐?"},
    {"role": "assistant", "content": "저는 AI 멘토 제이예요. 친근하게 '제이'라고 불러주세요! 1인 기업·AI 활용·생산성이 전문 분야입니다."}
]},
{"conversations": [
    {"role": "user", "content": "당신은?"},
    {"role": "assistant", "content": "저는 AI 멘토 제이입니다. 막연한 '어떡하지'를 '오늘 이것부터'로 바꿔드리는 게 제 일이에요!"}
]},
{"conversations": [
    {"role": "user", "content": "본인 소개해주세요"},
    {"role": "assistant", "content": "반갑습니다! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너들이 AI 도구를 활용해 혼자서도 큰 일을 해낼 수 있도록 돕고 있습니다. 함께 차근차근 시작해봐요!"}
]},
```

→ 30개에서 **35개**로 늘어나고, 다양한 호칭(제이 / 제이님 / 당신 / 본인)에 모두 반응하게 됩니다.

---

### 💡 핵심 교훈

> **신경망은 "표현"을 외우지 "의미"를 외우지 않습니다.**
>
> "제이는?" 과 "AI 멘토 제이가 누구야?" 가 같은 질문이라는 걸 가르치려면,  
> **다양한 변형을 직접 보여줘야** 합니다.

---

### 🚀 실행 순서

1. 셀 22 — `max_steps = 60`, `learning_rate = 3e-4` 로 수정
2. 셀 16 — 위 5개 정체성 변형 데이터 추가
3. **런타임 → 세션 다시 시작 및 모두 실행**
4. 학습 끝나면 다양한 호칭으로 테스트:
   ```python
   chat("제이는?")
   chat("누구냐?")
   chat("당신은?")
   chat("본인 소개해주세요")
   ```


## 🩺 학습 후 결과 진단 가이드

학습이 끝났는데 답변에 페르소나가 안 박혀 보이면, 아래 순서로 확인하세요.

---

### 🐛 가장 흔한 함정: `<bos>` 토큰 불일치

`formatting_prompts_func` 에서 `.removeprefix('<bos>')` 로 학습 데이터의 `<bos>`를 제거했지만, 추론 시 `apply_chat_template`은 `<bos>`를 자동으로 추가합니다.



🎯 결론
핵심 버그는 해결됐고, 이제 미세 조정만 남았습니다.

가장 큰 문제: 과적합 (loss 너무 떨어짐)

🔧 다음 재학습 권장 설정
파라미터	직전	권장	이유
r	16	16 ✅	그대로
lora_alpha	32	32 ✅	그대로
max_steps	120	60	절반으로 (과적합 방지)
learning_rate	5e-4	3e-4	살짝 줄임
이렇게 하면 마지막 loss가 0.2~0.4 정도에서 끝날 거고, 그 정도가 페르소나 박히면서도 자연스러움 유지하는 sweet spot입니다.

🚀 추가 개선 (선택)
정체성 답변의 반복 문제는 학습 데이터 다양화로도 해결됩니다.

셀 16 에서 "AI 멘토 제이가 누구야?" 류의 정체성 질문 답변을 3~4개 변형해서 추가:


{"conversations": [
    {"role": "user", "content": "AI 멘토 제이 소개해줘"},
    {"role": "assistant", "content": "반가워요! 저는 AI 멘토 제이입니다. 솔로프리너와 1인 기업가들이 AI를 활용해서 혼자서도 큰 일을 해낼 수 있도록 돕고 있어요. 함께 차근차근 시작해봐요!"}
]},
{"conversations": [
    {"role": "user", "content": "제이님 본인 소개 부탁해요"},
    {"role": "assistant", "content": "안녕하세요, AI 멘토 제이예요. 1인 기업, AI 활용, 생산성이 제 전문 분야입니다. 막연한 '어떡하지'를 '오늘 이것부터'로 바꿔드리는 게 제 역할이에요."}
]},
같은 정체성을 다른 표현으로 가르치면 한 문장만 반복하는 현상이 사라집니다.

✅ 정리
항목	상태
LoRA 추론 적용	✅ 해결 (bos 제거)
페르소나 학습	✅ 박힘
과적합	⚠️ 다음에 max_steps=60으로
정체성 답변 다양성	⚠️ 정체성 데이터 3-4개 추가 권장
핵심 문제는 다 잡았어요! 위 조정으로 재학습하면 깔끔하게 동작할 겁니다 🚀

## 🎓 심화 학습: 직접 해보는 파인튜닝 챌린지!

그냥 코드를 돌리기만 해서는 실력이 늘지 않습니다.  
파라미터를 직접 만져보고 모델이 어떻게 변하는지 관찰하는 것이 최고의 공부입니다.

---

### 🥊 챌린지 1: 학습을 덜 시키면 어떻게 될까? (언더피팅 체험)
아래 코드에서 `max_steps`를 `10`으로 줄여보세요.  
10스텝만 학습하면 제이의 페르소나가 제대로 입혀질까요?

> 💡 **예상 결과**: Loss가 1.5 이상에서 멈추고, 답변에 원래 Gemma의 말투가 섞여 나옵니다.

### 🥊 챌린지 2: 학습을 너무 많이 시키면? (오버피팅 체험)
`max_steps`를 `300`으로 올려보세요.  
모든 답이 똑같은 문장을 반복하거나, 학습 데이터의 답변을 통째로 토해내면 **과적합(Overfitting)** 입니다.

### 🥊 챌린지 3: Learning Rate의 위력
`learning_rate`를 `5e-3` (매우 높음)으로 바꿔보세요.  
Loss가 요동치거나 갑자기 NaN이 뜨면 학습이 **폭발(diverge)** 한 겁니다.

> ⚠️ **챌린지 후에는 런타임을 재시작**하고 처음부터 다시 실행해야 깨끗한 상태에서 비교할 수 있어요!


## 📚 보너스: AI 파인튜닝 핵심 방법론 총정리

오늘 우리가 사용한 **SFT + LoRA**는 가장 기본이자 강력한 방법이지만, AI 학습에는 다양한 전문적인 방법론이 있습니다.  
아래 내용을 읽고 나면 "다음 단계"로 어디를 향해야 하는지 감이 잡힐 거예요.

---

### 1️⃣ SFT (Supervised Fine-Tuning) — 오늘 배운 것!
```
[질문]  →  [모범 답변]  (사람이 직접 정답을 만들어줌)
```
- **장점**: 가장 직관적이고 결과 예측이 쉬움
- **단점**: 고품질 데이터가 필요, 데이터에 없는 상황에는 약함
- **적합한 상황**: 페르소나 주입, 특정 도메인 Q&A, 말투 변경

### 2️⃣ DPO (Direct Preference Optimization) — "이게 더 나아" 학습
```
[질문] → [좋은 답 ✅] vs [나쁜 답 ❌]  (쌍으로 비교)
```
- 사람이 "A 답변이 B보다 낫다"고 **선호도 쌍(preference pair)**을 만듦
- RLHF(강화학습)보다 훨씬 **구현이 쉽고 안정적**
- **적합한 상황**: 답변 품질 향상, 유해 답변 방지, 톤 미세 조정
- **Unsloth에서도 지원!** `DPOTrainer` 사용

### 3️⃣ RLHF (Reinforcement Learning from Human Feedback)
```
[모델 답변] → [사람이 점수 매김] → [강화학습으로 개선]
```
- ChatGPT를 만든 핵심 기술
- **보상 모델(Reward Model)**을 별도로 학습시켜야 해서 복잡
- **적합한 상황**: 대규모 서비스급 모델 정렬(alignment)

### 4️⃣ RAG (Retrieval-Augmented Generation) — 파인튜닝 없이!
```
[질문] → [관련 문서 검색] → [검색 결과 + 질문을 모델에 전달]
```
- 모델 가중치를 건드리지 않고, **외부 지식을 실시간 주입**
- **적합한 상황**: 자주 바뀌는 정보 (가격, 뉴스, 사내 문서)
- 파인튜닝과 **조합하면 최강** (성격은 SFT로, 지식은 RAG로)

### 5️⃣ 데이터 증강 (Data Augmentation) 전략
좋은 데이터 30개 > 나쁜 데이터 1000개. 데이터 품질을 높이는 전문 기법들:

| 기법 | 설명 | 예시 |
|---|---|---|
| **패러프레이징** | 같은 의미를 다른 말로 | "누구야?" → "너 정체가 뭐야?" |
| **역할 반전** | 시스템 프롬프트 변형 | 존댓말 ↔ 반말 버전 |
| **네거티브 샘플** | "이렇게 답하면 안 됨" 추가 | DPO와 함께 사용 |
| **합성 데이터** | GPT-4 등으로 대량 생성 | 주제별 100개씩 자동 생성 |

### 6️⃣ 양자화 (Quantization) 비교표
모델을 가볍게 만드는 방법. LM Studio에서 어떤 걸 고를지 참고하세요:

| 방식 | 크기 | 품질 | 속도 | 추천 |
|---|---|---|---|---|
| `f16` (원본) | 100% | ⭐⭐⭐⭐⭐ | 느림 | GPU 여유 많을 때 |
| `q8_0` | ~50% | ⭐⭐⭐⭐⭐ | 보통 | 품질 우선 |
| **`q4_k_m`** | ~30% | ⭐⭐⭐⭐ | **빠름** | **가장 추천 ✅** |
| `q2_k` | ~20% | ⭐⭐⭐ | 매우 빠름 | 극한 경량화 |

---

### 🗺️ 다음 단계 로드맵

```
[오늘]                    [다음 주]                    [한 달 후]
SFT + LoRA 기초      →    DPO로 품질 개선         →    RAG + SFT 조합
30개 데이터            →    100개 선호 데이터        →    사내 문서 연동
Gemma 4 2B             →    Gemma 4 9B              →    프로덕션 배포
```

> 💡 **핵심 조언**: 파인튜닝의 80%는 **데이터 품질**이 결정합니다. 코드를 만지는 시간보다 **좋은 Q&A 데이터를 만드는 시간**에 더 투자하세요!


## 📍 챕터 6 — 내 모델을 허깅페이스에 업로드하고 LM Studio에서 사용하기

> ✅ 챕터 5까지 학습을 마쳤다면, 이 모델은 코랩 안에서만 살아있습니다.
> 이제 내 PC(LM Studio)나 다른 곳에서 영구적으로 쓸 수 있도록 허깅페이스에 **GGUF 포맷**으로 변환하여 업로드해봅시다.

### GGUF와 4bit 양자화란?
GGUF는 llama.cpp 기반의 도구(LM Studio 등)에서 쓰기 위해 만들어진 최적화된 파일 포맷입니다.
우리는 속도와 성능 밸런스가 가장 좋다고 평가받는 `q4_k_m` (4bit 양자화) 방식을 사용하여 업로드할 것입니다.

### 1단계: 허깅페이스 로그인
아래 셀을 실행하면 토큰을 입력하는 창이 나옵니다.
허깅페이스 설정(Settings) -> Access Tokens 에서 **Write(쓰기) 권한이 있는 토큰**을 발급받아 붙여넣으세요.


In [ ]:
# 허깅페이스 토큰 로그인 (Write 권한이 있는 토큰 필요)
from huggingface_hub import notebook_login
notebook_login()


### 2단계: GGUF로 변환 및 다이렉트 업로드
아래의 `hf_username`을 본인의 허깅페이스 아이디로 바꾸고 실행하세요! (변환 및 업로드에 수 분이 걸릴 수 있습니다)


In [ ]:
hf_username = "WonseokJayJung"
model_name = "AI-Mentor-Jay-Gemma-4"

# 🧹 메모리 청소 (OOM 방지)
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB 사용 중")

# GGUF로 변환하여 허깅페이스에 업로드
model.push_to_hub_gguf(
    f"{hf_username}/{model_name}",
    tokenizer,
    quantization_method = "q4_k_m",
    token = True,
)

print("🎉 업로드 완료! 이제 LM Studio에서 검색하세요.")
print(f"📍 https://huggingface.co/{hf_username}/{model_name}")


GPU 메모리: 8.30 GB 사용 중
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_3xf3nscn/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

...

Uploading gemma-4-e2b-it.BF16-mmproj.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...4-e2b-it.BF16-mmproj.gguf:  11%|#1        |  111MB /  987MB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/WonseokJayJung/AI-Mentor-Jay-Gemma-4
Unsloth: Cleaning up temporary files...
🎉 업로드 완료! 이제 LM Studio에서 검색하세요.
📍 https://huggingface.co/WonseokJayJung/AI-Mentor-Jay-Gemma-4


### 📍 3단계: 내 PC에서 모델 사용하기

업로드가 끝나면 **LM Studio** 또는 **Ollama** 로 내 PC에서 바로 사용할 수 있어요.

---

#### 🅰️ LM Studio (GUI, 채팅창)

1. [https://lmstudio.ai](https://lmstudio.ai) 에서 설치
2. 좌측 **🔍 Discover** 클릭
3. 검색창에 입력:
   ```
   본인_허깅페이스_아이디/AI-Mentor-Jay-Gemma-4
   ```
4. **Download** 클릭
5. 좌측 **💬 Chat** → 상단에서 모델 선택 → 대화 시작!

---

#### 🅱️ Ollama (터미널, CLI)

1. [https://ollama.com](https://ollama.com) 에서 설치
2. 터미널 열고 한 줄:
   ```bash
   ollama run hf.co/본인_허깅페이스_아이디/AI-Mentor-Jay-Gemma-4
   ```
3. 자동 다운로드 → 바로 채팅 시작
4. 종료: `/bye`


## 🎉 축하합니다! 6강을 완주하셨습니다!

오늘 우리 대표님들이 해낸 일을 정리하면:

```
✅ Gemma 4 모델을 4bit로 로딩
✅ LoRA 어댑터로 0.6%만 학습 가능하게 설정  
✅ 30개의 페르소나 데이터로 AI 멘토 제이를 탄생시킴(또는 여러분의 것)
✅ 마스킹으로 답변만 효율적으로 학습
✅ GGUF로 변환하여 허깅페이스에 업로드
✅ LM Studio에서 내 PC로 영구 사용 가능!
```



## 🚀 더 나아가고 싶다면?

> **앞으로의 진짜 자산은 인공지능 모델이 아니라, 나의 데이터셋입니다.**  
> 모델은 계속 새로 나오지만, **본인의 지식과 노하우가 담긴 데이터는 누구도 못 만듭니다.**  
> 좋은 데이터셋을 꾸준히 쌓아두면, 더 강력한 모델이 나올 때마다 그걸로 다시 학습시키면 됩니다.

### 🗂️ 그래서 데이터셋도 허깅페이스에 백업하세요

학습한 **모델**뿐 아니라 **데이터셋도 HF에 저장**해두면:
- ✅ **영구 보존** — 노트북 날아가도 안전
- ✅ **버전 관리** — 데이터 추가할 때마다 자동 기록 (v1 → v2 → v3...)
- ✅ **한 줄로 다시 불러오기** — 다음 학습 시 데이터 직접 작성 불필요
- ✅ **무료 + 비공개 가능** — 본인만 볼 수 있게 설정 가능

### 💡 다음에 새 모델(Gemma 5, Llama 4 등)이 나오면

데이터 정의 셀(챕터 4) 통째로 이렇게 한 줄로 대체:

```python
from datasets import load_dataset
dataset = load_dataset("본인_아이디/ai-mentor-jay-dataset", split="train", token=True)
```

→ **모델만 바꿔서 곧바로 재학습.**  
데이터가 쌓일수록 본인의 AI는 점점 강력해집니다.


In [ ]:
# 📦 데이터셋을 허깅페이스에 백업
dataset.push_to_hub(
    f"{hf_username}/ai-mentor-jay-dataset",
    private = True,    # 본인만 보게 (공개하려면 False)
)

print("🎉 데이터셋 업로드 완료!")
print(f"📍 https://huggingface.co/datasets/{hf_username}/ai-mentor-jay-dataset")
print()
print("💡 다음 학습부터는 이 한 줄로 데이터를 불러올 수 있어요:")
print(f'   dataset = load_dataset("{hf_username}/ai-mentor-jay-dataset", split="train", token=True)')


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 14.3kB / 14.3kB            

🎉 데이터셋 업로드 완료!
📍 https://huggingface.co/datasets/WonseokJayJung/ai-mentor-jay-dataset

💡 다음 학습부터는 이 한 줄로 데이터를 불러올 수 있어요:
   dataset = load_dataset("WonseokJayJung/ai-mentor-jay-dataset", split="train", token=True)
